# 05 — Equity and spatial statistics (Phase 5)

Phase 4 answered *how much* each hex can reach. This notebook asks the
equity questions: **who** is underserved and **where**. Every statistic
weights hexes by their residents (a hex of 5,000 people counts 5,000
times) and covers inhabited city hexes only — an empty mountain cell
with zero access is not an injustice.

Read every number with four caveats in mind (they are generated into
`equity_report.json` with their figures):

- **Transit is approximate and a lower bound** — rebuilt from OSM route
  relations with assumed speeds and headways. Only routes with ordered
  stop members could be modelled, so for many residents the "transit"
  value is simply their walking value.
- **Car is free-flow** (no congestion) — an upper bound. The transit/car
  gap therefore overstates car dependence: read it as a ranking.
- **The population raster is skewed between districts**, so everything
  is computed under two weightings: the raster as it is, and the raster
  calibrated to the district shares of employed residents. Charts draw
  the raster run; where the weighting changes a figure, the calibrated
  value is printed with it.
- **Service measures live at the fine resolution** — at r8 a facility's
  hex centroid can be far from the facility itself.

By default this runs on the committed sample: a 2 km window with 3
coarse hexes and **synthetic** district statistics. Its coarse grid is
too small for local statistics, which take their documented "too few
hexes" path there and run at the fine resolution only. Set
`ALMATY_CONFIG=config/config.yaml` for the city figures.

In [ ]:
import json
import os
import re
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import patheffects
from matplotlib.colors import LinearSegmentedColormap, to_hex, to_rgb
from matplotlib.patches import Patch, PathPatch, Rectangle
from matplotlib.path import Path as MplPath

from almaty_access.access.build import build_measures, build_times
from almaty_access.config import load_config
from almaty_access.equity.build import (
    DESERTS_FILE,
    DISTRICTS_FILE,
    HEX_FILE_TEMPLATES,
    LORENZ_FILE,
    NEAREST_FILE,
    REPORT_FILE,
    SUMMARY_FILE,
    build_equity,
)
from almaty_access.equity.inequality import gini, weighted_quantile
from almaty_access.prepare.transit import prepare_transit

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent
CONFIG_PATH = Path(os.environ.get("ALMATY_CONFIG", REPO / "config" / "config.sample.yaml"))
config = load_config(CONFIG_PATH)
prepare_transit(config)  # all cached
build_times(config)
build_measures(config)
build_equity(config)

P = config.paths.processed
A, F = config.h3.analysis_resolution, config.h3.fine_resolution
RASTER, CALIBRATED = "raster", "calibrated"
report = json.loads((P / REPORT_FILE).read_text("utf-8"))
HAS_CALIBRATED = CALIBRATED in report["results"]
summary = pd.read_parquet(P / SUMMARY_FILE)
lorenz = pd.read_parquet(P / LORENZ_FILE)
districts = pd.read_parquet(P / DISTRICTS_FILE)
deserts = pd.read_parquet(P / DESERTS_FILE)
nearest = pd.read_parquet(P / NEAREST_FILE)
access = {res: pd.read_parquet(P / f"access_r{res}.parquet") for res in (A, F)}
grid = {res: gpd.read_parquet(P / f"grid_r{res}.parquet")[["h3", "geometry"]] for res in (A, F)}


def hex_layers(res: int, weighting: str = RASTER) -> gpd.GeoDataFrame:
    table = pd.read_parquet(P / HEX_FILE_TEMPLATES[weighting].format(res=res))
    return grid[res].merge(table, on="h3")


hexes = {res: hex_layers(res) for res in (A, F)}
district_shapes = gpd.read_parquet(config.paths.interim / "boundary_districts.parquet")
HEADLINE_A = list(config.equity.headline_analysis)
HEADLINE_F = list(config.equity.headline_fine)


def block(res: int, weighting: str = RASTER) -> dict:
    return report["results"][weighting][f"r{res}"]


def pick(table: pd.DataFrame, res: int, weighting: str = RASTER, measure: str | None = None):
    rows = table[(table["res"] == res) & (table["weights"] == weighting)]
    return rows if measure is None else rows[rows["measure"] == measure]


print(f"config: {CONFIG_PATH.name}")
for res in (A, F):
    pop = report["population"][f"r{res}"]
    print(
        f"r{res}: {pop['inhabited_hexes']:,} inhabited hexes, "
        f"{pop['inhabited_population']:,.0f} residents on the raster"
    )
print("weightings:")
for name, text in report["weightings"].items():
    print(f"  {name}: {text}")
print()
print("HEADLINE:", report["headline"]["sentence"])
for line in report["companions"]:
    print("  -", line)

## One small chart system

All figures below share one set of rules so they read as a family:
the form is chosen by the data's job, **one hue (blue) carries
magnitude**, the **blue ↔ red pair carries cold/hot polarity** with a
neutral grey for "nothing significant", and distinct hues appear only
where the four modes share one axis — in a fixed order, checked for
colour-vision-deficiency separation, and always with direct labels.
Text stays in ink; a colour never has to be read alone, because the
table behind every chart is printed under it.

In [ ]:
SURFACE, INK, INK_2, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, NEUTRAL = "#e1e0d9", "#c3c2b7", "#f0efec"
BLUE, ORANGE, AQUA, YELLOW, RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948"
LIGHT_BLUE = "#86b6ef"
# Fixed hue order = the config's mode order; validated as an adjacent palette.
MODE_COLOR = {"transit": BLUE, "car": ORANGE, "bike": AQUA, "walk": YELLOW}
SEQ_BLUE = LinearSegmentedColormap.from_list(
    "seq_blue", ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
)
# 3x3 bivariate scheme (Stevens "pink-blue"): lightness falls along both axes.
BIVARIATE = {
    "1-1": "#e8e8e8",
    "1-2": "#ace4e4",
    "1-3": "#5ac8c8",
    "2-1": "#dfb0d6",
    "2-2": "#a5add3",
    "2-3": "#5698b9",
    "3-1": "#be64ac",
    "3-2": "#8c62aa",
    "3-3": "#3b4994",
}

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "font.family": "sans-serif",
        "font.size": 9.5,
        "text.color": INK,
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.titlesize": 10.5,
        "axes.titleweight": "semibold",
        "axes.titlelocation": "left",
        "axes.edgecolor": AXIS,
        "axes.linewidth": 1.0,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelcolor": INK_2,
        "ytick.labelcolor": INK_2,
        "grid.color": GRID,
        "grid.linewidth": 1.0,
        "grid.linestyle": "-",
        "legend.frameon": False,
        "legend.fontsize": 9,
    }
)


def blend(color: str, alpha: float) -> str:
    """The colour at the given opacity over the chart surface, as a hex."""
    top, base = np.array(to_rgb(color)), np.array(to_rgb(SURFACE))
    return to_hex(alpha * top + (1 - alpha) * base)


def style(ax, grid: str | None = "y") -> None:
    """Recessive chrome: no box, hairline solid grid behind the marks."""
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0)
    ax.set_axisbelow(True)
    if grid:
        ax.grid(axis=grid)


def measure_label(measure: str) -> str:
    job = re.fullmatch(r"accpct_jobs_(\w+)_(\d+)", measure)
    if job:
        return f"{job.group(1).capitalize()} \u00b7 {job.group(2)} min"
    service = re.fullmatch(r"sfca_(\w+)_walk", measure)
    if service:
        return f"{service.group(1).capitalize()} (2SFCA)"
    return measure


def bars(ax, labels, values, color=BLUE, fmt="{:.1f}", texts=None) -> None:
    """Ranked horizontal bars: thin, square at the baseline, a 4 px
    rounded data end, the value printed at the tip."""
    values = np.asarray(values, dtype=float)
    positions = np.arange(len(values))[::-1]
    top = np.nanmax(values) if len(values) and np.nanmax(values) > 0 else 1.0
    ax.set_xlim(0, top * 1.3)
    ax.set_ylim(-0.6, len(values) - 0.4)
    ax.figure.canvas.draw()
    px_x = ax.bbox.width / (top * 1.3)
    px_y = ax.bbox.height / len(values) if len(values) else 1.0
    thickness = min(0.62, 22.0 / px_y)
    codes = [
        MplPath.MOVETO,
        MplPath.LINETO,
        MplPath.CURVE3,
        MplPath.CURVE3,
        MplPath.LINETO,
        MplPath.CURVE3,
        MplPath.CURVE3,
        MplPath.LINETO,
        MplPath.CLOSEPOLY,
    ]
    for i, (y, value) in enumerate(zip(positions, values, strict=True)):
        if not np.isfinite(value) or value <= 0:
            continue
        low, high = y - thickness / 2, y + thickness / 2
        rx, ry = min(4.0 / px_x, value / 2), min(4.0 / px_y, thickness / 2)
        outline = [
            (0, low),
            (value - rx, low),
            (value, low),
            (value, low + ry),
            (value, high - ry),
            (value, high),
            (value - rx, high),
            (0, high),
            (0, low),
        ]
        ax.add_patch(PathPatch(MplPath(outline, codes), linewidth=0, facecolor=color))
        text = fmt.format(value) if texts is None else texts[i]
        ax.text(value + top * 0.02, y, text, va="center", ha="left", color=INK, fontsize=9)
    ax.set_yticks(positions, labels)
    ax.spines[["top", "right", "bottom"]].set_visible(False)
    ax.tick_params(length=0)
    ax.set_xticks([])


def base_map(ax, frame: gpd.GeoDataFrame, title: str) -> None:
    """District outlines and names for orientation; no axes."""
    district_shapes.boundary.plot(ax=ax, color=AXIS, linewidth=0.8, zorder=3)
    bounds = frame.total_bounds
    for row in district_shapes.itertuples():
        point = row.geometry.representative_point()
        if bounds[0] <= point.x <= bounds[2] and bounds[1] <= point.y <= bounds[3]:
            label = ax.text(
                point.x,
                point.y,
                str(row.district).capitalize(),
                fontsize=8,
                color=INK,
                ha="center",
                va="center",
                zorder=4,
            )
            label.set_path_effects([patheffects.withStroke(linewidth=2.5, foreground=SURFACE)])
    pad_x, pad_y = (bounds[2] - bounds[0]) * 0.03, (bounds[3] - bounds[1]) * 0.03
    ax.set_xlim(bounds[0] - pad_x, bounds[2] + pad_x)
    ax.set_ylim(bounds[1] - pad_y, bounds[3] + pad_y)
    ax.set_title(title)
    ax.set_axis_off()


def edge_width(res: int) -> float:
    """A hairline surface gap between hexes, thinner on the fine grid."""
    return 0.35 if res == A else 0.12

## First, the weights: is the population raster itself fair?

Everything below weights hexes by residents, so the residents deserve a
check of their own. The raster (Kontur Population) is a model; the one
residence-based series in the pipeline is the labour-force survey's
count of **employed residents per district**. Comparing each district's
share of the two shows the raster shortchanging the dense high-rise
districts and overweighting the low-rise periphery — and the dense
districts are the best served, so the skew inflates inequality.

Phase 5 does not rewrite the population layer (that would change every
earlier phase). It runs the whole analysis **twice**: on the raster,
and on residents rescaled so district totals follow the survey shares.
Measures that contain residents themselves — 2SFCA, where they are the
demand competing for each facility — are *recomputed*, not just
re-weighted. The table shows what moves.

(On the committed sample the district statistics are synthetic test
numbers, not a survey; the chart legend and the printed source say so.)

In [ ]:
calibration = report["calibration"]
if not HAS_CALIBRATED:
    print("calibrated run skipped:", calibration["skipped_reason"])
else:
    # The fine grid assigns hexes to districts most precisely; its
    # factors are the ones the report's caveats quote.
    grid_key = f"r{F}" if f"r{F}" in calibration["district_shares"] else f"r{A}"
    shares = pd.DataFrame(calibration["district_shares"][grid_key]).T * 100
    shares = shares.sort_values("target", ascending=False)
    target_label = (
        "SYNTHETIC test shares"
        if calibration["synthetic"]
        else "employed residents (district statistics)"
    )
    fig, ax = plt.subplots(figsize=(7.2, 0.44 * len(shares) + 1.5))
    rows = np.arange(len(shares))[::-1]
    ax.hlines(rows, shares[RASTER], shares["target"], color=AXIS, linewidth=2, zorder=1)
    ax.scatter(
        shares[RASTER],
        rows,
        s=64,
        color=LIGHT_BLUE,
        edgecolor=SURFACE,
        linewidth=2,
        zorder=2,
        label="population raster (Kontur)",
    )
    ax.scatter(
        shares["target"],
        rows,
        s=64,
        color=BLUE,
        edgecolor=SURFACE,
        linewidth=2,
        zorder=3,
        label=f"{target_label}, {calibration['year']}",
    )
    ax.set_yticks(rows, [name.capitalize() for name in shares.index])
    ax.set_ylim(-0.7, len(shares) - 0.3)
    ax.set_xlim(0, float(shares.to_numpy().max()) * 1.12)
    ax.set_xlabel("share of the city total, %")
    ax.set_title(
        f"District shares of residents: raster vs calibration target ({grid_key} grid)", pad=30
    )
    ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=2, borderaxespad=0.2)
    style(ax, grid="x")
    plt.tight_layout()
    plt.show()

    shares["raster / target"] = shares[RASTER] / shares["target"]
    print(
        f"calibration target: {calibration['indicator']} {calibration['year']}; "
        f"source: {calibration['source']}"
        + ("  [SYNTHETIC test numbers, not real statistics]" if calibration["synthetic"] else "")
    )
    print(shares.round(2).to_string())
    for res, measures in ((A, HEADLINE_A), (F, HEADLINE_F)):
        table = []
        for measure in measures:
            entry = report["comparison"][f"r{res}"][measure]
            row = {"measure": measure}
            for key in ("mean", "gini"):
                row[f"{key} raster"] = entry[key][RASTER]
                row[f"{key} calibrated"] = entry[key][CALIBRATED]
            row["deserts raster"] = entry["deserts"][RASTER]
            row["deserts calibrated"] = entry["deserts"][CALIBRATED]
            row["deserts both"] = entry["deserts"]["both"]
            table.append(row)
        print(f"\nr{res}: raster vs calibrated")
        print(pd.DataFrame(table).to_string(index=False))
    moved = {
        m: e["district_ranks_changed"]
        for m, e in report["comparison"][f"r{F}"].items()
        if e["district_ranks_changed"]
    }
    print("\ndistrict ranks that change with the weighting:", moved if moved else "none")

## Lorenz curves, Gini and Palma — how unequal is job access?

Line the residents up from the worst to the best served and accumulate
both people and accessibility: that is the **Lorenz curve**. If everyone
had the same access it would follow the diagonal; the further it sags,
the more unequal the distribution. The **Gini coefficient** is twice the
shaded area (0 = perfect equality, values near 1 = a few people hold
almost everything). The **Palma ratio** compares the tails instead: the
access held by the best-served 10% of residents over that held by the
worst-served 40% (0.25 under perfect equality). The curve is drawn for
the raster; the title gives the calibrated Gini next to it.

In [ ]:
fig, axes = plt.subplots(1, len(HEADLINE_A), figsize=(3.3 * len(HEADLINE_A), 3.9), sharey=True)
axes = np.atleast_1d(axes)
for ax, measure in zip(axes, HEADLINE_A, strict=True):
    curve = pick(lorenz, A, RASTER, measure)
    row = pick(summary, A, RASTER, measure).iloc[0]
    ax.plot([0, 1], [0, 1], color=AXIS, linewidth=1)
    if curve["access_share"].notna().all():
        ax.fill_between(
            curve["pop_share"],
            curve["access_share"],
            curve["pop_share"],
            color=BLUE,
            alpha=0.10,
            linewidth=0,
        )
        ax.plot(
            curve["pop_share"],
            curve["access_share"],
            color=BLUE,
            linewidth=2,
            solid_capstyle="round",
        )
    gini_text = "n/a" if np.isnan(row["gini"]) else f"{row['gini']:.2f}"
    if HAS_CALIBRATED and not np.isnan(row["gini"]):
        other = pick(summary, A, CALIBRATED, measure).iloc[0]["gini"]
        gini_text += f" ({other:.2f} calibrated)"
    palma_text = (
        "n/a"
        if np.isnan(row["palma"])
        else ("\u221e" if np.isinf(row["palma"]) else f"{row['palma']:.2f}")
    )
    ax.set_title(f"{measure_label(measure)}\nGini {gini_text}\nPalma {palma_text}")
    ax.set_aspect("equal")
    ax.set_xticks([0, 0.5, 1], ["0", "50%", "100%"])
    ax.set_yticks([0, 0.5, 1], ["0", "50%", "100%"])
    style(ax, grid="both")
axes[0].set_ylabel("share of all job access")
axes[0].text(
    0.5,
    0.54,
    "perfect equality",
    rotation=45,
    rotation_mode="anchor",
    color=MUTED,
    fontsize=8,
    ha="center",
    va="bottom",
)
fig.supxlabel("share of residents, worst served first", fontsize=9.5, color=INK_2)
fig.suptitle(
    f"Who holds the job access? (r{A}, % of city jobs reachable)",
    x=0.01,
    y=0.99,
    ha="left",
    fontsize=11.5,
    fontweight="semibold",
)
plt.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

table = summary[(summary["res"] == A) & summary["measure"].isin(HEADLINE_A)]
print(
    table[["weights", "measure", "mean", "median", "p10", "p90", "gini", "palma"]]
    .round(3)
    .to_string(index=False)
)

## Districts — the same question, by administrative area

Population-weighted mean of each measure per district. All four panels
keep the district order of the first one, so a district that drops down
a panel is doing worse on that mode than its overall position suggests.
For job measures these means do not depend on the weighting at all:
calibration rescales whole districts, and a district's own mean is
unchanged by it.

In [ ]:
order = pick(districts, A, RASTER, HEADLINE_A[0]).sort_values("rank")["district"].tolist()
fig, axes = plt.subplots(
    1, len(HEADLINE_A), figsize=(3.3 * len(HEADLINE_A), 0.42 * len(order) + 1.4), sharey=True
)
axes = np.atleast_1d(axes)
for ax, measure in zip(axes, HEADLINE_A, strict=True):
    rows = pick(districts, A, RASTER, measure).set_index("district").reindex(order)
    bars(ax, [name.capitalize() for name in order], rows["mean"].to_numpy())
    ax.set_title(measure_label(measure))
fig.suptitle(
    f"Mean % of city jobs reachable, by district (r{A}, population-weighted)",
    x=0.01,
    ha="left",
    fontsize=11.5,
    fontweight="semibold",
)
plt.tight_layout()
plt.show()

pivot = (
    pick(districts, A, RASTER)
    .pivot(index="district", columns="measure", values="mean")
    .reindex(order)[HEADLINE_A]
)
print(pivot.round(2).to_string())

## How much of "transit" is transit? — and the mode gap

Before reading any transit map, look at how far the modelled network
reaches. `pop_share_transit_equals_walk` is the share of residents for
whom the transit layer adds **nothing** over walking within the
threshold: where OSM lists a route's ways but not its stops, the model
has nothing to board. Transit rankings and "transit deserts" therefore
partly measure where the mapping is thin.

`modegap_transit_car_T` divides jobs reachable by transit by jobs
reachable by car within the same *T* minutes — a **lower bound** over an
**upper bound**. The level overstates car dependence everywhere; the
*pattern*, where transit gets closest to the car, is the usable part.

In [ ]:
for weighting in report["results"]:
    coverage = block(A, weighting).get("transit_coverage")
    if not coverage:
        continue
    same = coverage["pop_share_transit_equals_walk"]
    print(f"[{weighting}] share of residents with transit job access = walking job access:")
    print("   ", {f"{t} min": f"{v:.0%}" for t, v in same.items()})
    print(
        f"    residents in hexes with a modelled route: "
        f"{coverage['pop_share_in_hexes_with_modelled_routes']:.0%}; in hexes with mapped "
        f"stop POIs but no modelled route: "
        f"{coverage['pop_share_in_hexes_with_stop_pois_but_no_route']:.0%}"
    )
    twin = coverage.get("headline_walk_only")
    if twin:
        key = "pop_share" if weighting == RASTER else "pop_share_calibrated"
        print(
            f"    headline share on WALKING alone ({twin['measure']}): {twin['pop_share']:.1%} "
            f"vs {report['headline'][key]:.1%} with modelled transit"
        )

thresholds = list(config.travel.thresholds_min)
gap_t = 45 if 45 in thresholds else thresholds[len(thresholds) // 2]
gap_col = f"modegap_transit_car_{gap_t}"
frame = hexes[A]
shown = frame[frame["inhabited"] & frame[gap_col].notna()]

fig, ax = plt.subplots(figsize=(7.4, 6.6))
upper = float(np.nanquantile(shown[gap_col], 0.98)) if len(shown) else 1.0
upper = upper if upper > 0 else 1.0
shown.plot(
    column=gap_col,
    ax=ax,
    cmap=SEQ_BLUE,
    vmin=0,
    vmax=upper,
    edgecolor=SURFACE,
    linewidth=edge_width(A),
    legend=True,
    legend_kwds={
        "label": f"transit / car job access within {gap_t} min (top 2% clipped)",
        "shrink": 0.62,
    },
)
base_map(ax, frame[frame["in_city"]], f"Transit / car job-access ratio within {gap_t} min (r{A})")
plt.show()

for weighting in report["results"]:
    print(f"\n[{weighting}] transit / car ratio over residents")
    print(pd.DataFrame(block(A, weighting)["mode_gap"]).T.to_string())

## LISA — statistically significant cold and hot spots

Global Moran's I says accessibility is spatially clustered — it always
is: neighbouring hexes share most of what they can reach. The useful
tool is the **local** statistic. For each hex, LISA asks whether the hex
*and its neighbours* are jointly low (a **cold spot**, LL) or jointly
high (a **hot spot**, HH) beyond what reshuffling the other hexes would
produce; HL / LH mark outliers that differ from their surroundings.

Does the population weighting matter here? For the **job measures** the
cluster *locations* do not depend on it: the statistic is computed on
hex values, and those contain no residents — only the share of
residents living inside the clusters changes. The **2SFCA measures**
are different: residents are their demand term, the calibrated run
recomputes the values, and the hot spots themselves move. The second
table prints the cluster counts under both weightings and how many
hexes are hot under both.

Three implementation notes that matter for trust. P-values are
two-sided and **tie-aware**: on zero-inflated measures one of the
p-values the library offers can leave a whole measure without a single
significant cell, and the other calls every zero region a cold spot.
One test per hex means multiple testing — the table also counts the
clusters that survive false-discovery-rate control (shown as
"unresolved" where the permutation count cannot decide). And a
permutation p-value is itself an **estimate**: the `borderline` column
counts the hexes within two standard errors of the 0.05 line —
significant ones another seed could drop (−) and non-significant ones
it could add (+). Where that column is large (the zero-heavy 2SFCA
measures, whose hexes flip in blocks) read the cluster counts as rough,
and part of any raster-vs-calibrated difference as noise.

In [ ]:
LISA_COLOR = {
    "HH": RED,
    "LL": BLUE,
    "HL": blend(RED, 0.45),
    "LH": blend(BLUE, 0.45),
    "ns": NEUTRAL,
    "island": NEUTRAL,
}
LISA_NAME = {
    "HH": "Hot spot (high among high)",
    "LL": "Cold spot (low among low)",
    "HL": "High outlier among low",
    "LH": "Low outlier among high",
    "ns": "Not significant",
}

panels = [(A, HEADLINE_A[0]), (F, HEADLINE_F[0])]
fig, axes = plt.subplots(1, 2, figsize=(14.5, 6.8))
for ax, (res, measure) in zip(axes, panels, strict=True):
    frame = hexes[res]
    domain = frame[frame["inhabited"]]
    entry = block(res)["lisa"][measure]
    labels = domain[f"lisa_{measure}"].fillna("ns")
    domain.plot(
        ax=ax, color=labels.map(LISA_COLOR).tolist(), edgecolor=SURFACE, linewidth=edge_width(res)
    )
    base_map(ax, frame[frame["in_city"]], f"{measure_label(measure)} (r{res})")
    if "skipped_reason" in entry:
        ax.text(
            0.5,
            0.02,
            "LISA skipped: " + entry["skipped_reason"],
            transform=ax.transAxes,
            ha="center",
            va="bottom",
            fontsize=8.5,
            color=INK_2,
        )
        continue
    counts = entry["clusters"]
    shown_keys = [
        key for key in ("HH", "LL", "HL", "LH", "ns") if counts.get(key, 0) or key in ("HH", "LL")
    ]
    handles = [
        Patch(
            facecolor=LISA_COLOR[key],
            edgecolor=GRID,
            linewidth=0.5,
            label=f"{LISA_NAME[key]} ({counts.get(key, 0):,})",
        )
        for key in shown_keys
    ]
    ax.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.0),
        ncol=3,
        fontsize=8.5,
        columnspacing=1.4,
        handlelength=1.4,
    )
fig.suptitle(
    "Where low access clusters: LISA cold spots (blue) and hot spots (red)",
    x=0.01,
    ha="left",
    fontsize=11.5,
    fontweight="semibold",
)
plt.tight_layout()
plt.show()


def fdr_text(entry: dict) -> str:
    count = entry["fdr"]["significant"]
    return "unresolved" if count is None else str(count)


def borderline_text(entry: dict) -> str:
    margin = entry["borderline"]
    return f"-{margin['significant']}/+{margin['not_significant']}"


rows, both_rows = [], []
for res, measures in ((A, HEADLINE_A), (F, HEADLINE_F)):
    for measure in measures:
        entry = block(res)["lisa"][measure]
        if "skipped_reason" in entry:
            rows.append({"res": res, "measure": measure, "note": entry["skipped_reason"]})
            continue
        rows.append(
            {
                "res": res,
                "measure": measure,
                "moran_I": entry["global_moran"]["I"],
                "zero_hexes": entry["zero_share_hexes"],
                "HH": entry["clusters"].get("HH", 0),
                "LL": entry["clusters"].get("LL", 0),
                "borderline": borderline_text(entry),
                "fdr": fdr_text(entry),
                "cold_pop": entry["cold_spot_pop_share"],
                "hot_pop": entry["hot_spot_pop_share"],
            }
        )
        if not HAS_CALIBRATED:
            continue
        other = block(res, CALIBRATED)["lisa"][measure]
        if "skipped_reason" in other:
            continue
        overlap = report["comparison"][f"r{res}"][measure].get("hot_spot_hexes", {})
        both_rows.append(
            {
                "res": res,
                "measure": measure,
                "HH raster": entry["clusters"].get("HH", 0),
                "HH calibrated": other["clusters"].get("HH", 0),
                "HH both": overlap.get("both"),
                "LL raster": entry["clusters"].get("LL", 0),
                "LL calibrated": other["clusters"].get("LL", 0),
                "fdr raster": fdr_text(entry),
                "fdr calibrated": fdr_text(other),
                "hot_pop raster": entry["hot_spot_pop_share"],
                "hot_pop calibrated": other["hot_spot_pop_share"],
            }
        )
print("raster run:")
print(pd.DataFrame(rows).to_string(index=False))
print(
    "borderline = hexes (any label) within two standard errors of p = 0.05: significant "
    "ones another seed could drop (-), non-significant ones it could add (+)"
)
if both_rows:
    print(
        "\nwhat the weighting changes (identical cluster counts = identical hexes for the "
        "job measures; 2SFCA values are recomputed, so its clusters move):"
    )
    print(pd.DataFrame(both_rows).to_string(index=False))

## Access deserts — dense places among the worst served

A desert is a **dense** hex (population above the 75th percentile of
inhabited hexes) whose residents are in the **worst-served quarter of
the population** (access at or below the population-weighted 25th
percentile). The README's literal rule takes the access percentile
across *hexes* instead; that threshold is set by near-empty outskirts,
and for the coarse job measures it flags (almost) nothing — its count is
printed for comparison.

Both halves of the rule use residents, so the list depends on the
weighting. The map separates deserts that hold under **both**
weightings (the robust core) from those that appear under only one. For
the transit measure, remember the previous section: most of these hexes
have no modelled route at all, so their "transit" access is a walking
value.

In [ ]:
measure = HEADLINE_A[0]
frame = hexes[A]
domain = frame[frame["inhabited"]].copy()
flag_raster = domain[f"desert_{measure}"].to_numpy()
if HAS_CALIBRATED:
    other = hex_layers(A, CALIBRATED).set_index("h3")[f"desert_{measure}"]
    flag_calibrated = other.reindex(domain["h3"]).to_numpy(dtype=bool)
else:
    flag_calibrated = flag_raster
both = flag_raster & flag_calibrated
one = flag_raster ^ flag_calibrated

fig, ax = plt.subplots(figsize=(7.4, 6.6))
colors = np.where(both, ORANGE, np.where(one, blend(ORANGE, 0.4), NEUTRAL))
domain.plot(ax=ax, color=colors.tolist(), edgecolor=SURFACE, linewidth=edge_width(A))
base_map(ax, frame[frame["in_city"]], f"Access deserts for {measure_label(measure).lower()} (r{A})")
ranked = pick(deserts, A, RASTER, measure)
robust = ranked[ranked["h3"].isin(set(domain.loc[both, "h3"]))]
for slot, row in enumerate(robust.head(3).itertuples()):
    point = frame.loc[frame["h3"] == row.h3, "geometry"].iloc[0].representative_point()
    ax.annotate(
        f"#{row.rank}  {row.population:,.0f} residents",
        (point.x, point.y),
        xytext=(0.0, 0.93 - 0.055 * slot),
        textcoords="axes fraction",
        fontsize=8.5,
        color=INK,
        ha="left",
        va="center",
        arrowprops={
            "arrowstyle": "-",
            "color": INK_2,
            "linewidth": 0.8,
            "shrinkA": 2,
            "shrinkB": 2,
            "relpos": (1.0, 0.5),
        },
        zorder=5,
    )
handles = [Patch(facecolor=ORANGE, label=f"Desert under both weightings ({int(both.sum())})")]
if HAS_CALIBRATED:
    handles.append(
        Patch(
            facecolor=blend(ORANGE, 0.4),
            label=f"Desert under one weighting only ({int(one.sum())})",
        )
    )
handles.append(
    Patch(facecolor=NEUTRAL, edgecolor=GRID, linewidth=0.5, label="Other inhabited hexes")
)
ax.legend(handles=handles, loc="lower left", fontsize=8.5)
plt.show()

for weighting in report["results"]:
    entry = block(A, weighting)["deserts"][measure]
    print(
        f"[{weighting}] {entry['n_hexes']} hexes, {entry['population']:,.0f} residents "
        f"({entry['pop_share']:.1%}); dense = more than {entry['pop_threshold']:,.0f}; "
        f"worst-served quarter = at most {entry['access_threshold']:g}% of city jobs; "
        f"README-literal rule: {entry['n_hexes_literal_rule']} hexes"
    )
    if "hexes_with_modelled_routes" in entry:
        print(
            f"    of these, {entry['hexes_with_modelled_routes']} have a modelled route and "
            f"{entry['hexes_with_stop_pois']} contain mapped stop POIs"
        )
    print("    by district:", {k: f"{v:,.0f}" for k, v in entry["population_by_district"].items()})
quiet = [m for m in HEADLINE_A if block(A)["deserts"][m]["low_spread"]]
print("no deserts reported (no meaningful low tail):", quiet if quiet else "none")
print(
    ranked.head(10)[["rank", "h3", "district", "population", "value"]]
    .round(2)
    .to_string(index=False)
)

## Bivariate map — population and access together

Each inhabited hex is classed twice: population tercile across hexes
(few → many residents) and access tercile across **residents** (little
→ much — weighted, for the same reason the desert rule is). The 3×3
scheme darkens along both axes: **magenta** is the equity-critical
corner (a dense hex among the worst-served third of residents), **teal**
its opposite, dark blue the dense well-served core. Hexes with identical
values always share a class, so class sizes are unequal by design.

In [ ]:
measure = HEADLINE_A[0]
frame = hexes[A]
domain = frame[frame["inhabited"]]
classes = domain[f"biv_{measure}"]

fig, ax = plt.subplots(figsize=(7.4, 6.6))
domain.plot(
    ax=ax, color=classes.map(BIVARIATE).tolist(), edgecolor=SURFACE, linewidth=edge_width(A)
)
base_map(
    ax, frame[frame["in_city"]], f"Residents and {measure_label(measure).lower()} job access (r{A})"
)
legend = ax.inset_axes([0.03, 0.04, 0.2, 0.2])
for key, color in BIVARIATE.items():
    pop_class, access_class = (int(part) for part in key.split("-"))
    legend.add_patch(
        Rectangle(
            (access_class - 1, pop_class - 1),
            1,
            1,
            facecolor=color,
            edgecolor=SURFACE,
            linewidth=1.5,
        )
    )
legend.set_xlim(0, 3)
legend.set_ylim(0, 3)
legend.set_xticks([])
legend.set_yticks([])
legend.set_xlabel("access \u2192", fontsize=8.5, color=INK_2, labelpad=2)
legend.set_ylabel("residents \u2192", fontsize=8.5, color=INK_2, labelpad=2)
for spine in legend.spines.values():
    spine.set_visible(False)
plt.show()

table = domain.groupby(classes)["population"].agg(hexes="size", residents="sum")
table["resident_share"] = table["residents"] / table["residents"].sum()
print(table.round({"residents": 0, "resident_share": 3}).to_string())
print(
    "\n'3-1' = a dense hex among the worst-served third of residents; '1-3' = few "
    "residents, plenty of access."
)

## Services on foot — independent of the transit layer

These numbers use only the walking network, so the transit caveat does
not touch them. Minutes run to the nearest facility *point* at the fine
resolution; a resident with no walkable path to any facility counts as
beyond every threshold rather than disappearing from the statistic.
Where the two weightings print different shares, the bar shows the
raster and the label gives the range.

In [ ]:
city = pick(nearest, F, RASTER)
city = city[city["district"] == "ALL"]
quoted = 15 if 15 in set(city["threshold_min"]) else int(city["threshold_min"].min())
ranked = city[city["threshold_min"] == quoted].sort_values("pop_share_within", ascending=False)
texts = [f"{share:.0%}" for share in ranked["pop_share_within"]]
if HAS_CALIBRATED:
    other = pick(nearest, F, CALIBRATED)
    other = other[(other["district"] == "ALL") & (other["threshold_min"] == quoted)]
    other = other.set_index("facility")["pop_share_within"]
    texts = []
    for row in ranked.itertuples():
        low, high = sorted((row.pop_share_within, other[row.facility]))
        same = f"{low:.0%}" == f"{high:.0%}"
        texts.append(f"{row.pop_share_within:.0%}" if same else f"{low:.0%}\u2013{high:.0%}")

fig, ax = plt.subplots(figsize=(6.6, 0.42 * len(ranked) + 1.0))
bars(
    ax,
    [name.capitalize() for name in ranked["facility"]],
    ranked["pop_share_within"].to_numpy() * 100,
    texts=texts,
)
ax.set_title(f"Residents within a {quoted}-minute walk of the nearest... (r{F})")
plt.tight_layout()
plt.show()

print("share of residents within T minutes (city, raster):")
print(
    city.pivot(index="facility", columns="threshold_min", values="pop_share_within")
    .round(3)
    .to_string()
)
by_district = pick(nearest, F, RASTER)
by_district = by_district[
    (by_district["district"] != "ALL") & (by_district["threshold_min"] == quoted)
]
print(f"\nshare within {quoted} minutes, by district (unchanged by the calibration):")
print(
    by_district.pivot(index="district", columns="facility", values="pop_share_within")
    .round(2)
    .to_string()
)
print("\nwalking minutes to the nearest facility (population-weighted, raster):")
print(pd.DataFrame(block(F)["nearest"]).T.to_string())
point = report["population"][f"r{F}"]
print(
    f"\nnote: {point['point_mass_hexes']} r{F} hexes ({point['point_mass_population']:,.0f} "
    "residents) are flagged `point_mass` - a dasymetric artifact that tops fine-resolution "
    "rankings by residents."
)

## The centre–periphery gradient

The one socio-economic proxy the open data supports is distance to the
historic centre (housing type would need real floor counts, and 97% of
the floors in the building layer are defaults). All four modes share one
axis because they share one unit — % of city jobs reachable.

In [ ]:
bands = pd.DataFrame(block(A)["center_gradient"])
fig, ax = plt.subplots(figsize=(8.2, 4.4))
x = np.arange(len(bands))
for measure in HEADLINE_A:
    job = re.fullmatch(r"accpct_jobs_(\w+)_(\d+)", measure)
    if not job:
        continue
    mode = job.group(1)
    values = pd.to_numeric(bands[measure], errors="coerce").to_numpy()
    ax.plot(
        x,
        values,
        color=MODE_COLOR[mode],
        linewidth=2,
        marker="o",
        markersize=7,
        markeredgecolor=SURFACE,
        markeredgewidth=2,
        solid_capstyle="round",
        label=measure_label(measure),
    )
    if np.isfinite(values[0]):
        ax.annotate(
            measure_label(measure),
            (x[0], values[0]),
            xytext=(-10, 0),
            textcoords="offset points",
            ha="right",
            va="center",
            fontsize=9,
            color=INK,
        )
ax.set_xticks(x, bands["band"])
ax.set_xlim(-1.25, len(bands) - 0.7)
ax.set_ylim(0, 105)
ax.set_ylabel("% of city jobs reachable (population-weighted mean)")
ax.set_xlabel("distance from the historic centre")
ax.set_title(f"Job access by distance from the historic centre (r{A})", pad=30)
ax.legend(
    loc="lower left",
    bbox_to_anchor=(0.0, 1.0),
    ncol=4,
    columnspacing=1.4,
    handlelength=1.6,
    borderaxespad=0.2,
)
style(ax)
plt.tight_layout()
plt.show()

print("raster residents:")
print(bands.to_string(index=False))
if HAS_CALIBRATED:
    print("\ncalibrated residents:")
    print(pd.DataFrame(block(A, CALIBRATED)["center_gradient"]).to_string(index=False))
print(
    "\nSpearman rank correlation with distance to the centre (across hexes, so the same "
    "under both weightings):",
    block(A)["center_spearman"],
)

## Verdict

The cell below recomputes the headline numbers from the raw tables with
independent code and refuses to finish if anything disagrees with the
report.

In [ ]:
failures = []


def check(name: str, condition: bool) -> None:
    print(("PASS " if condition else "FAIL ") + name)
    if not condition:
        failures.append(name)


first = HEADLINE_A[0]
features = pd.read_parquet(P / f"features_r{A}.parquet", columns=["h3", "pop_residents"])
raw = access[A].merge(features, on="h3")
raw = raw[raw["in_city"] & (raw["pop_residents"] >= config.equity.min_population)]
weights = raw["pop_residents"].to_numpy()
values = raw[first].to_numpy()

# 1. The one-sentence result, recomputed from the access table.
cutoff = config.equity.headline_cutoff_pct
share = float(weights[values < cutoff].sum() / weights.sum())
check(
    "headline share reproduced from raw tables", abs(share - report["headline"]["pop_share"]) < 1e-3
)
check(
    "headline sentence carries the transit caveat", "lower bound" in report["headline"]["sentence"]
)
if HAS_CALIBRATED:
    stats = pd.read_csv(config.paths.raw / "district_stats.csv")
    survey = stats[
        (stats["indicator"] == "employed_residents")
        & (stats["year"] == report["calibration"]["year"])
    ]
    survey = survey.groupby("district")["value"].mean()
    raster_totals = raw.groupby("district")["pop_residents"].sum()
    survey = survey.reindex(raster_totals.index)
    factor = (survey / survey.sum()) / (raster_totals / raster_totals.sum())
    calibrated = weights * raw["district"].map(factor).to_numpy(dtype=float)
    check(
        "calibration preserves the resident total",
        abs(calibrated.sum() - weights.sum()) < 1e-6 * weights.sum(),
    )
    share_calibrated = float(calibrated[values < cutoff].sum() / calibrated.sum())
    check(
        "calibrated headline share reproduced independently",
        abs(share_calibrated - report["headline"]["pop_share_calibrated"]) < 1e-3,
    )
    stored = hex_layers(A, CALIBRATED).set_index("h3")["population"].reindex(raw["h3"])
    check(
        "calibrated residents in the hex file match the recomputation",
        bool(np.allclose(stored.to_numpy(), calibrated, rtol=1e-3)),
    )

# 2. Gini by brute force (no Lorenz code): mean absolute difference.
order = np.argsort(values)
v, w = values[order], weights[order]
cum_w = np.cumsum(w)
mean_abs_diff = 2 * np.sum(w * v * (2 * cum_w - w - cum_w[-1])) / cum_w[-1] ** 2
brute = mean_abs_diff / (2 * np.sum(w * v) / cum_w[-1]) if np.sum(w * v) > 0 else float("nan")
stored = float(pick(summary, A, RASTER, first)["gini"].iloc[0])
check(
    "Gini matches an independent formula",
    (np.isnan(brute) and np.isnan(stored)) or abs(brute - stored) < 1e-6,
)
check("library Gini agrees too", np.isnan(stored) or abs(gini(values, weights) - stored) < 1e-9)

# 3. Lorenz curves are valid curves.
valid = True
for _, curve in lorenz.groupby(["res", "weights", "measure"]):
    curve_shares = curve["access_share"].to_numpy()
    if np.isnan(curve_shares).all():
        continue
    valid &= abs(curve_shares[0]) < 1e-9 and abs(curve_shares[-1] - 1) < 1e-9
    valid &= bool((np.diff(curve_shares) >= -1e-12).all())
check("Lorenz curves start at 0, end at 1 and never decrease", bool(valid))

# 4. District tables cover every inhabited resident.
covered = all(
    abs(group["population"].sum() - report["results"][weighting][f"r{res}"]["population"]) <= 1
    for (res, weighting, _), group in districts.groupby(["res", "weights", "measure"])
)
check("district populations add up to the inhabited population", covered)

# 5. The desert rule, recomputed (including the no-low-tail guard: the
#    worst-served quarter's MEAN against the median, counted resident by
#    resident from the sorted hexes of check 2).
pop_cut = float(np.quantile(weights, config.equity.desert_pop_quantile))
access_cut = weighted_quantile(values, weights, config.equity.desert_access_quantile)
median = weighted_quantile(values, weights, 0.5)
quarter = config.equity.desert_access_quantile * cum_w[-1]
inside_quarter = np.clip(quarter - (cum_w - w), 0, w)  # residents of each hex in that quarter
worst_mean = float((inside_quarter * v).sum() / quarter)
has_tail = median <= 0 or worst_mean / median < 0.95
expected = set(raw.loc[(weights > pop_cut) & (values <= access_cut), "h3"]) if has_tail else set()
listed = set(pick(deserts, A, RASTER, first)["h3"])
check("desert hexes match the rule recomputed here", expected == listed)

# 6. LISA labels and counts.
allowed = {"HH", "LL", "HL", "LH", "ns", "island"}
labels_ok, counts_ok = True, True
for res, measures in ((A, HEADLINE_A), (F, HEADLINE_F)):
    domain = hexes[res][hexes[res]["inhabited"]]
    for measure in measures:
        entry = block(res)["lisa"][measure]
        labels = domain[f"lisa_{measure}"]
        if "skipped_reason" in entry:
            labels_ok &= bool(labels.isna().all())
            continue
        labels_ok &= set(labels.dropna()) <= allowed and bool(labels.notna().all())
        counts_ok &= labels.value_counts().to_dict() == entry["clusters"]
check("LISA labels are valid (or absent where LISA was skipped)", bool(labels_ok))
check("LISA cluster counts match the report", bool(counts_ok))
if HAS_CALIBRATED:
    same_places, overlap_ok = True, True
    for res, measures in ((A, HEADLINE_A), (F, HEADLINE_F)):
        first_run = hexes[res].set_index("h3")
        second_run = hex_layers(res, CALIBRATED).set_index("h3").reindex(first_run.index)
        for measure in measures:
            one_way = first_run[f"lisa_{measure}"].fillna("")
            other_way = second_run[f"lisa_{measure}"].fillna("")
            if measure.startswith("accpct_jobs_"):
                same_places &= bool((one_way == other_way).all())
            counted = report["comparison"][f"r{res}"][measure].get("hot_spot_hexes")
            if counted:
                overlap_ok &= counted == {
                    RASTER: int((one_way == "HH").sum()),
                    CALIBRATED: int((other_way == "HH").sum()),
                    "both": int(((one_way == "HH") & (other_way == "HH")).sum()),
                }
    check(
        "job-measure LISA clusters sit on the same hexes under both weightings", bool(same_places)
    )
    check("hot-spot overlap counts match the two per-hex files", bool(overlap_ok))

# 7. Bivariate classes and nearest-facility shares.
domain = hexes[A][hexes[A]["inhabited"]]
check(
    "every inhabited hex has a bivariate class",
    bool(domain[f"biv_{first}"].str.fullmatch(r"[123]-[123]").all()),
)
monotone = all(
    group.sort_values("threshold_min")["pop_share_within"].is_monotonic_increasing
    for _, group in nearest.groupby(["res", "weights", "district", "facility"])
)
check("a longer walk never reaches fewer residents", bool(monotone))

# 8. 2SFCA under calibrated demand still conserves supply.
if HAS_CALIBRATED:
    fine_raster = hex_layers(F, RASTER).set_index("h3")
    fine_calibrated = hex_layers(F, CALIBRATED).set_index("h3")
    conserved = True
    for column in (c for c in fine_calibrated.columns if c.startswith("sfca_")):
        original = access[F].set_index("h3")[column].reindex(fine_raster.index).to_numpy()
        before = float((original * fine_raster["population"].to_numpy()).sum())
        after = float(
            (fine_calibrated[column].to_numpy() * fine_calibrated["population"].to_numpy()).sum()
        )
        conserved &= before == 0 or abs(after - before) <= 0.02 * before
    check("2SFCA recomputed with calibrated demand conserves supply", bool(conserved))

# 9. The report is strict JSON and names its caveats.
json.dumps(report, allow_nan=False)
check(
    "report states the transit lower bound",
    any("LOWER BOUND" in caveat for caveat in report["caveats"]),
)

print()
if failures:
    raise AssertionError(f"verdict failed: {failures}")
print("All checks passed.")
print("RESULT:", report["headline"]["sentence"])